# 04 — Batching Benchmark

> Companion to **Week 11**, Part 4 of the lecture.

## The idea in one line

> Calling a model **once** with 100 examples is almost always faster than
> calling it **100 times** with one example each.

This is because every model call has fixed overhead: allocating memory,
launching kernels, Python function calls, etc. That cost is paid *per call*,
not *per example*. Batch your inputs and you pay it once.

Think of it like an Uber Pool: the same driver, more passengers per trip.


## Step 1 — Build a small model

We will use the same model for every experiment so the only thing changing
is the batch size.


In [ ]:
import time
import pandas as pd
import torch
import torch.nn as nn

torch.manual_seed(42)

model = nn.Sequential(
    nn.Linear(256, 512),
    nn.ReLU(),
    nn.Linear(512, 512),
    nn.ReLU(),
    nn.Linear(512, 10),
)
model.eval()
print("Model built. Parameters:", sum(p.numel() for p in model.parameters()))


## Step 2 — Time the model at different batch sizes

For each batch size we:

1. Build a random input tensor of that size.
2. Warm up (a few calls so any one-time setup doesn't pollute timing).
3. Run the model many times and average.
4. Record latency per batch, latency per example, and throughput.


In [ ]:
batch_sizes = [1, 8, 32, 128, 512]
rows = []

for batch_size in batch_sizes:
    x = torch.randn(batch_size, 256)

    # Warm-up
    for _ in range(20):
        with torch.no_grad():
            model(x)

    # Time it
    start = time.perf_counter()
    runs = 200
    for _ in range(runs):
        with torch.no_grad():
            model(x)
    avg_ms = (time.perf_counter() - start) * 1000 / runs

    rows.append(
        {
            "batch_size":           batch_size,
            "avg_batch_ms":         avg_ms,
            "avg_per_example_ms":   avg_ms / batch_size,
            "examples_per_second":  batch_size / (avg_ms / 1000),
        }
    )

results = pd.DataFrame(rows)
results


### What to look for

- `avg_batch_ms` grows slowly with batch size — not linearly.
- `avg_per_example_ms` should drop sharply as the batch grows.
- `examples_per_second` (throughput) should grow with batch size.

That is the whole point of batching: more work per call = lower per-example
latency = higher throughput.

## Step 3 — Plot it


In [ ]:
results.plot(
    x="batch_size",
    y=["avg_per_example_ms", "examples_per_second"],
    subplots=True,
    figsize=(8, 6),
    grid=True,
    title=["Latency per example", "Throughput"],
)


## 🧪 Your turn

1. Add a batch size of `2048` to the list. Does throughput keep growing, or
   does it level off?
2. Replace the model with a much bigger one (e.g. hidden size 2048). How does
   the relationship between batch size and per-example latency change?

> 💡 **In production**, the right batch size is the **largest one your memory
> can handle, while still meeting your latency budget.**
